# Machine Learning
### Bernoulli naïve Bayes

**Bernoulli naïve Bayes** (**Bernoulli NB**) is a probabilistic **classifier** that
- Each feature $x_i$ is binary: $x_i\in \{0,1\}$
- It uses the Naïve Bayes assumption that features are **conditionally independent** given the class. Thus:
  - $P(x_1,x_2,...,x_q|C=k)=\prod_{i=1}^{q}P(x_i|C=k)$ 

The model estimates class **priors** and feature **likelihoods** (via maximum likelihood with smoothing) to compute the **posterior** probability of each class for a given sample. 
<br> Bernoulli NB is widely used in text classification (e.g., spam detection) where features are binary indicators of word presence.
<hr>

Bernoulli NB is composed of two algorithms or phases: **Training** and **prediction**:
- **Training algorithm**: Given feature matrix $X$ (each row holds features of one sample) and corresponding class labels $\boldsymbol{y}$, we compute prior probailities $P(C=k)$ and likelihood of each feature to be "1" given the class, which is denoted by $P(x_i=1|C=k)$ or $\theta_{k,i}$:
  - $P(C=k)=\frac{\text{number of samples with } C=k}{\text{total number of samples}}$
  - $P(x_i=1|C=k)=\theta_{k,i}=\frac{1+\text{ number of samples where } C=k \text{ and } x_i=1}{2+ \text{ number of samples where } C=k}$
- **Prediction algorithm**: For the new feature vector $\boldsymbol{x}_{new}$, we choose the class with maximum **posterior**:
  - $C_{predicted}=arg max_k P(C=k|\boldsymbol{x}_{new})$
  - where $P(C=k|\boldsymbol{x}_{new})\propto P(\boldsymbol{x}_{new}|C=k)\cdot P(C=k)$
  - such that $P(\boldsymbol{x}_{new}|C=k)=\prod_{i=1}^q \theta_{k,i}^{x_i} (1-\theta_{k,i}^{(1-x_i)})$ 
<hr>

**Hint:** Some points on Bernoulli NB:
- Bernoulli NB assumes binary features — not continuous.
- Works well with text data (word presence/absence).
- Sensitive to feature selection — good features are key.
- Often used with **bag-of-words** representation.

<hr>

**Reminder:** We have computed here the class posterior probability by Bayes\' theorem:
<br> $P(C=k|\boldsymbol{x})=\frac{P(\boldsymbol{x}|C=k) \cdot P(C=k)}{P(\boldsymbol{x})}$
<br> Since the denominator $P(\boldsymbol{x})$ is constant for all classes, we ignore it for classification, and only compare the numerators in the prediction algorithm.
<hr>


In the following:
- We implement **Bernoulli NB** from scratch in Python. 
- Afterwards, we create a synthetic dataset holding presence/absence of words in emails divided into two classes: {Spam, NotSpam}. 
- Then, our Bernoulli NB is used for **spam detection** based on the given dataset.

<hr>

https://github.com/ostad-ai/Machine-Learning
<br> Explanation: https://www.pinterest.com/HamedShahHosseini/Machine-Learning/

In [1]:
# Import required modules
import numpy as np
import pandas as pd

In [2]:
class BernoulliNaiveBayes:
    def __init__(self):
        self.priors = {}  # P(C=k)
        self.likelihoods = {}  # θ_{k,i} for each class k and feature i

    def fit(self, X, y):
        """
        Fit the Bernoulli Naive Bayes model.
        X: 2D array of binary features (n_samples, n_features)
        y: 1D array of class labels (n_samples)
        """
        n_samples, n_features = X.shape
        self.n_features = n_features
        self.classes = np.unique(y)

        # Compute prior probabilities: P(C=k)
        self.priors = {cls: np.sum(y == cls) / n_samples for cls in self.classes}

        # Initialize likelihoods: θ_{k,i} = P(x_i=1 | C=k)
        self.likelihoods = {cls: {} for cls in self.classes}
        for cls in self.classes:
            # Total samples in class cls
            total_class = np.sum(y == cls)
            for i in range(n_features):
                # Count: number of samples with class=cls and feature i=1
                count = np.sum((y == cls) * (X[:, i] == 1))                
                # Laplace smoothing: add 1 to numerator, 2 to denominator
                self.likelihoods[cls][i] = (count + 1) / (total_class + 2)

    def predict(self, X):
        """
        Predict class for new samples.
        Returns: list of predicted class labels
        """
        n_samples = X.shape[0]
        predictions = []
        for x in X:
            # Compute posterior for each class
            posteriors = {}
            for cls in self.classes:
                # P(C=cls | x) ∝ P(x|C=cls) * P(C=cls)
                likelihood = 1.0
                for i in range(self.n_features):
                    if x[i] == 1:
                        likelihood *= self.likelihoods[cls][i]
                    else:
                        likelihood *= (1 - self.likelihoods[cls][i])
                posterior = likelihood * self.priors[cls]
                posteriors[cls] = posterior

            # Choose class with highest posterior
            predicted_class = max(posteriors, key=posteriors.get)
            predictions.append(predicted_class)
        return predictions

    def predict_proba(self, X):
        """
        Returns probability for each class for each sample.
        """
        n_samples = X.shape[0]
        proba = []
        for x in X:
            posteriors = {}
            for cls in self.classes:
                likelihood = 1.0
                for i in range(self.n_features):
                    if x[i] == 1:
                        likelihood *= self.likelihoods[cls][i]
                    else:
                        likelihood *= (1 - self.likelihoods[cls][i])
                posterior = likelihood * self.priors[cls]
                posteriors[cls] = posterior
            proba.append(posteriors)
        return proba

In [3]:
# --- Example Usage ---
print("=== Bernoulli Naive Bayes Classifier (From Scratch) ===")

# Create sample data (spam detection)
data = {
    'email_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'label': [0, 0, 0, 1, 1, 1, 0, 1, 1, 1],  # 0: not spam, 1: spam
    'free': [0, 0, 0, 1, 0, 0, 0, 1, 0, 1],   # 1 = "free" word present
    'urgent': [0, 0, 0, 0, 1, 0, 0, 0, 1, 0], # 1 = "urgent" word present
    'money': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], # 1 = "money" word present
    'click': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], # 1 = "click" word present
    'offer': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]  # 1 = "offer" word present
}

df = pd.DataFrame(data)
X = df[['free', 'urgent', 'money', 'click', 'offer']].values  # Binary features
y = df['label'].values

# Train the model
nb = BernoulliNaiveBayes()
nb.fit(X, y)

# Predict on training data (for evaluation)
y_pred = nb.predict(X)
print("Predictions:", y_pred)
print("Actual labels:", y)
print("Accuracy:", np.mean(y_pred == y))

# Predict on new sample
new_email = np.array([[1, 0, 0, 0, 0]])  # "free=1, others=0"
prediction = nb.predict(new_email)
print("\nPrediction (class) for new email:", prediction)

# Predict probabilities
probabilities = nb.predict_proba(new_email)
print("\nProbabilities for new email:")
for i, prob in enumerate(probabilities):
    print(f"Sample {i+1}: {prob}")

# Optional: Print model parameters for debugging
print("\n=== Model Parameters ===")
for cls in nb.classes:
    print(f"Class {cls}:")
    for i, feat in enumerate(nb.likelihoods[cls]):
        print(f"  Feature {i}: P(x_{i}=1 | C={cls}) = {nb.likelihoods[cls][feat]:.4f}")
    print(f"  Prior P(C={cls}) = {nb.priors[cls]:.4f}")

=== Bernoulli Naive Bayes Classifier (From Scratch) ===
Predictions: [0, 0, 0, 1, 1, 1, 0, 1, 1, 1]
Actual labels: [0 0 0 1 1 1 0 1 1 1]
Accuracy: 1.0

Prediction (class) for new email: [1]

Probabilities for new email:
Sample 1: {0: 0.03215020576131688, 1: 0.0791015625}

=== Model Parameters ===
Class 0:
  Feature 0: P(x_0=1 | C=0) = 0.1667
  Feature 1: P(x_1=1 | C=0) = 0.1667
  Feature 2: P(x_2=1 | C=0) = 0.1667
  Feature 3: P(x_3=1 | C=0) = 0.1667
  Feature 4: P(x_4=1 | C=0) = 0.1667
  Prior P(C=0) = 0.4000
Class 1:
  Feature 0: P(x_0=1 | C=1) = 0.5000
  Feature 1: P(x_1=1 | C=1) = 0.3750
  Feature 2: P(x_2=1 | C=1) = 0.2500
  Feature 3: P(x_3=1 | C=1) = 0.2500
  Feature 4: P(x_4=1 | C=1) = 0.2500
  Prior P(C=1) = 0.6000
